# Explore: price + levels + signals + trades

Interactive view of the strategy. Edit the parameters, re-run cells, roam freely.
Everything here calls the same modules as the CLI — no parallel logic.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from sfp_reversion.backtest.engine import BacktestParams, run_backtest
from sfp_reversion.data.loader import load_ohlc
from sfp_reversion.levels.detection import detect_levels
from sfp_reversion.report import metrics_table
from sfp_reversion.signals.decision_tree import DecisionTreeParams, generate_signals

In [ ]:
# ---- parameters: edit and re-run everything below ----
PAIR = "EUR_USD"
START = "2025-01-01"
END = "2025-12-31"
VIEW_BARS = 1000  # hourly bars shown on the chart (tail of the window)

hourly = load_ohlc(PAIR, granularity="H1")
daily = load_ohlc(PAIR)
print(len(hourly), "hourly bars,", len(daily), "daily bars")

In [ ]:
params = DecisionTreeParams.from_config()
levels = detect_levels(
    daily,
    lookback=params.swing_lookback,
    cluster_ticks=params.cluster_ticks,
    tick_size=params.tick_size,
    atr_period=params.atr_period,
    min_swing_magnitude_atr=params.min_swing_magnitude_atr,
)
print(len(levels), "daily levels")
pd.DataFrame(
    [(round(lv.price, 5), lv.kind, lv.touches, str(lv.formed_at.date())) for lv in levels[-10:]],
    columns=["price", "kind", "touches", "formed"],
)

In [ ]:
sig = generate_signals(hourly, daily, params)
window = sig[(sig["timestamp"] >= START) & (sig["timestamp"] < END)]
print(len(window), "signals in window")
window.head(20)

In [ ]:
import matplotlib.pyplot as plt

view = hourly.loc[START:END].tail(VIEW_BARS)
vsig = window[window["timestamp"] >= str(view.index[0])]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(view.index, view["close"], color="black", linewidth=0.8, label="close")
for lv in levels:
    if view["low"].min() <= lv.price <= view["high"].max():
        ax.axhline(lv.price, color="blue" if lv.kind == "support" else "red",
                   alpha=0.4, linewidth=1)
longs = vsig[vsig["direction"] == "long"]
shorts = vsig[vsig["direction"] == "short"]
ax.scatter(longs["timestamp"], longs["limit_price"], marker="^", color="green", s=60, label="long")
ax.scatter(shorts["timestamp"], shorts["limit_price"], marker="v", color="red", s=60, label="short")
ax.set_title(f"{PAIR} hourly with levels + signals")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
bt = BacktestParams.from_config()
res = run_backtest(hourly, sig, bt)
trades = res.trades_df
wtrades = trades[(trades["signal_ts"] >= START) & (trades["signal_ts"] < END)]
print(metrics_table(wtrades, res.equity_curve, bt.min_equity).to_string(index=False))
wtrades[["entry_ts", "direction", "entry_price", "exit_price", "pnl", "exit_reason"]]